# Inspect Consolidated Snow-Covered Area Dataset

Load and visualize the MOD10C1 v061 daily SCA fields for early March 2000.

Sources (see `catalog/variables.yml` → `snow_covered_area`):
- MOD10C1 v061 `Day_CMG_Snow_Cover` — 0-100 percent snow per cell (with flag values 107, 237, 239, 250, 253, 255 that must be masked)
- MOD10C1 v061 `Day_CMG_Clear_Index` — 0-100 percent clear-sky per cell. **This is the variable TM 6-B10 calls "confidence interval"** for SCA filtering (CI > 70%). In v006 it was named `Day_CMG_Confidence_Index`; renamed in v061. Same flag values as snow cover.

The consolidated NetCDF also carries `Day_CMG_Cloud_Obscured` (kept for QA cross-checks) and `Snow_Spatial_QA` (a 0-4 categorical QA flag, *not* the CI — see the explanation cell below for why we don't load it here).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_TIME = "2000-03-01"
TARGET_YEAR = 2000


## Load the dataset

In [ ]:
import numpy as np
import xarray as xr

mod10c1_path = DATASTORE / "mod10c1_v061" / f"mod10c1_v061_{TARGET_YEAR}_consolidated.nc"


def _mask_flags(da):
    """Mask MOD10C1 flag values (>100) to NaN. Applies to percent-encoded
    variables: Day_CMG_Snow_Cover, Day_CMG_Clear_Index, Day_CMG_Cloud_Obscured.
    """
    return da.where((da >= 0) & (da <= 100))


datasets = {
    "MOD10C1 v061 (Day_CMG_Snow_Cover)": {
        "path": mod10c1_path,
        "var": "Day_CMG_Snow_Cover",
        "units": "percent (0-100, flag values masked)",
        "cmap": "Blues",
        "mask": _mask_flags,
    },
    "MOD10C1 v061 (Day_CMG_Clear_Index)": {
        "path": mod10c1_path,
        "var": "Day_CMG_Clear_Index",
        "units": "percent (0-100, flag values masked) - TM 6-B10 CI",
        "cmap": "viridis",
        "mask": _mask_flags,
    },
}

opened = {}
missing_vars = []
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run fetch first)")
        continue
    ds = xr.open_dataset(nc_path)
    if info["var"] not in ds.data_vars:
        missing_vars.append((label, info["var"]))
        ds.close()
        continue
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")

if missing_vars:
    print()
    print("WARN: variables missing from the consolidated NC:")
    for label, var in missing_vars:
        print(f"  - {label} ({var})")
    print("  Re-run `pixi run fetch-mod10c1 -- --project-dir <project> --period 2000/<end>`")
    print("  to pick up Day_CMG_Clear_Index / Day_CMG_Cloud_Obscured (added to catalog).")


## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)


## Plot March 1, 2000

In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(1, n, figsize=(8 * n, 6), squeeze=False)

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[0, idx]
        var = info["var"]
        da = info["mask"](ds[var].sel(time=TARGET_TIME, method="nearest"))
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap=info.get("cmap", "viridis"), robust=True)
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=10)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    fig.suptitle(f"MOD10C1 v061 — nearest to {TARGET_TIME}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


### Why we don't load `Snow_Spatial_QA` here (and why an earlier version of this repo did)

TM 6-B10 (Hay et al., 2023) describes the SCA calibration target as follows:

> "The daily snow-covered area values have an associated confidence interval, where a confidence interval equal to 100 percent indicates clear sky conditions with the highest level of confidence... daily values of snow-covered area were used if the confidence interval was greater than 70 percent."

The "100 percent = clear sky" phrasing maps to MOD10C1's **clear-sky percentage** variable, not to `Snow_Spatial_QA`:

- **`Day_CMG_Clear_Index`** (v061; was `Day_CMG_Confidence_Index` in v006) is a 0-100 percent variable. The value is the percentage of the cell that was cloud-free on that day. A value of 100 means the cell was fully clear-sky - exactly the TM 6-B10 wording. **This is what we use as the CI.**
- **`Snow_Spatial_QA`** is a 0-4 categorical *quality* flag (0=best, 1=good, 2=ok, 3=poor, 4=other) describing the algorithmic confidence in the snow retrieval, not the clear-sky percentage. Despite the source HDF carrying a `units: percent` attribute, the data are not on a 0-100 scale and dividing by 100 produces meaningless "fractions".

This repo previously had `range_method: modis_ci` configured to read `ci = Snow_Spatial_QA / 100` and filter `ci > 0.70`. That filter would *only* pass values >=70 on a 0-4 scale - i.e. only the special-case flag codes (237=inland water, 239=ocean, 250=cloud-obscured, 253=not mapped, 255=fill). Every legitimate QA value (0-4) would be rejected. The catalog has been corrected to read the CI from `Day_CMG_Clear_Index` instead.

`Snow_Spatial_QA` is still consolidated into the NetCDF for any future QA cross-checks (e.g. confirming that low Clear_Index correlates with poor QA flags), but is intentionally not loaded here - there is no quantitative use for it in the SCA target as defined by TM 6-B10.

**Practical note on flag values**

`Day_CMG_Snow_Cover`, `Day_CMG_Clear_Index`, and `Day_CMG_Cloud_Obscured` all carry the same flag values for non-data cells:

- 107 = lake ice
- 111 = night
- 237 = inland water
- 239 = ocean
- 250 = cloud-obscured water
- 253 = data not mapped
- 255 = fill

Any quantitative use must mask values outside the documented `valid_range` (0-100) before averaging or thresholding. Without the mask, the domain mean of `Day_CMG_Snow_Cover` for a typical CONUS day is around 100, dominated by 237/239/250 flag codes rather than real percent-snow values.

**For the SCA target builder**

`targets/sca.py` (currently a stub) should:

1. Load `Day_CMG_Snow_Cover` and `Day_CMG_Clear_Index`, masking flag values >100 to NaN.
2. Compute `sca = Day_CMG_Snow_Cover / 100` and `ci = Day_CMG_Clear_Index / 100`.
3. Discard any `(time, lat, lon)` cell where `ci <= 0.70`.
4. Aggregate to HRUs via gdptools using `sca` only on cells passing the CI filter.
5. Derive lower/upper bounds from sca and ci per the report's "error bound based on the daily SCA value and the associated confidence interval" - exact formula remains unconfirmed (PRMSobjfun.f not public).


## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()
